In [ ]:
# Import kaggle api
!pip install -q kaggle

In [ ]:
# Upload kaggle api file
from google.colab import files
files.upload()

In [ ]:
# Make directory for kaggle data
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download the data
!kaggle datasets download -d taejinwoo/multiwoz-22

In [ ]:
# Unzip the data
!unzip -o multiwoz-22.zip

In [ ]:
# List files for checks
!ls -R

In [ ]:
# Dialogues Loader
import os
import json

train_dir = "/content/MultiWOZ_2.2/train"

dialogues = []

for file in sorted(os.listdir(train_dir)):
    if file.endswith(".json"):
        with open(os.path.join(train_dir, file), "r") as f:
            dialogues.extend(json.load(f))

print(f"Loaded {len(dialogues)} dialogues.")

In [ ]:
# Preprocesser
conversation_pairs = []

for dialogue in dialogues :
  context = []
  for turn in dialogue["turns"]:
    speaker = turn["speaker"]
    utterance = turn["utterance"]

    if speaker == "SYSTEM":
      conversation_pairs.append({
                "dialogue_id": dialogue["dialogue_id"],
                "services": dialogue["services"],
                "context": context.copy(),
                "response": utterance
            })
    context.append(f"{speaker} : {utterance}")

print(f"Created {len(conversation_pairs)} training examples.")

In [ ]:
import random

example = random.choice(conversation_pairs)

print("Dialogue ID:", example["dialogue_id"])
print("Services:", example["services"])

print("\nContext:\n")
print("\n".join(example["context"]))

print("\nResponse:\n")
print(example["response"])

In [ ]:
# Dataset
# Tokenizer

from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

class MultiWOZDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length=256):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):

        sample = self.examples[idx]

        context = "\n".join(sample["context"])
        response = sample["response"]

        input_encoding = self.tokenizer(
            context,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        response = (
            self.tokenizer.bos_token +
            response +
            self.tokenizer.eos_token
        )

        target_encoding = self.tokenizer(
            response,
            max_length=64,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        return {
            "ctx_ids": input_encoding["input_ids"].squeeze(0),
            "ctx_mask": input_encoding["attention_mask"].squeeze(0),
            "rsp_ids": target_encoding["input_ids"].squeeze(0)
        }

In [ ]:
from torch.utils.data import DataLoader

train_dataset = MultiWOZDataset(conversation_pairs, tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True
)

In [ ]:
batch = next(iter(train_loader))

print(batch["ctx_ids"].shape)
print(batch["ctx_mask"].shape)
print(batch["rsp_ids"].shape)

In [ ]:
"""
Finsler geodesic encoder vs attention encoder for MultiWOZ 2.2 response generation.

Design: both encoders feed the SAME decoder and SHARE a tied embedding table, so any
difference in validation perplexity is attributable to the encoder, not to capacity or
vocabulary handling. Both produce the same number of memory slots (T // STRIDE), so the
decoder's cross-attention budget is identical.

    context tokens  --[encoder]-->  memory [B, T/STRIDE, Dmem]
                                        |
    response tokens --[decoder w/ cross-attention]--> next-token logits

The geodesic encoder integrates m velocities along the DERIVED Finsler geodesic and
emits the state (x, v) at each integration step as a memory slot. The attention encoder
runs causal self-attention and strided-pools to the same number of slots.

Usage (Colab):
    !pip -q install transformers
    python finsler_seq2seq.py build --mwoz-dir /content/MultiWOZ_2.2 --tokenizer gpt2
    python finsler_seq2seq.py run --encoder geodesic  --seed 0 --steps 2000
    python finsler_seq2seq.py run --encoder attention --seed 0 --steps 2000
    python finsler_seq2seq.py summary

If `transformers` is unavailable, `--tokenizer word` builds a word-level vocabulary from
the data instead; everything else is unchanged.
"""

import argparse, glob, json, math, os, re, time
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================
# CONFIG
# ============================================================================
DATA     = 'mwoz_seq2seq.pt'
RESULTS  = 'seq2seq_results.json'

CTX_LEN  = 64      # context tokens
RSP_LEN  = 64       # response tokens
D        = 64       # model width (embeddings, decoder)

# --- construction hyperparameters
NV       = 8        # manifold dimension n
MVEL     = 4        # number of velocities => m parallel geodesics
STRIDE   = 8        # tokens absorbed per geodesic integration step
DTAU     = 0.3      # d(tau) integration step
VCLAMP   = 3.0      # velocity norm ceiling (numerical stabilisation)
TILT     = 0.4      # ||c||, Randers tilt magnitude; 0 => Riemannian
DMEM     = 2 * MVEL * NV          # memory slot width = 64
NSLOT    = CTX_LEN // STRIDE      # memory slots = 32

# --- decoder
DEC_LAYERS = 2
DEC_HEADS  = 4


# ============================================================================
# TOKENIZERS  (pluggable: GPT-2 in Colab, word-level fallback offline)
# ============================================================================
class WordTokenizer:
    """Minimal word-level tokenizer built from the corpus."""
    name = 'word'

    def __init__(self, texts=None, cap=8000, state=None):
        if state is not None:
            self.itos = state['itos']
        else:
            import collections
            c = collections.Counter()
            for t in texts:
                c.update(self._split(t))
            self.itos = ['<pad>', '<unk>', '<bos>', '<eos>'] + [w for w, _ in c.most_common(cap)]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id, self.bos_id, self.eos_id = 0, 2, 3
        self.vocab_size = len(self.itos)

    @staticmethod
    def _split(t):
        return re.findall(r"[a-z0-9']+|[.,?!:]", t.lower())

    def encode(self, text, max_len, add_bos_eos=False):
        ids = [self.stoi.get(w, 1) for w in self._split(text)]
        if add_bos_eos:
            ids = [self.bos_id] + ids[:max_len - 2] + [self.eos_id]
        ids = ids[-max_len:] if not add_bos_eos else ids[:max_len]
        mask = [1] * len(ids) + [0] * (max_len - len(ids))
        ids = ids + [self.pad_id] * (max_len - len(ids))
        return ids, mask

    def state(self):
        return {'itos': self.itos}


class HFTokenizer:
    """Wraps a HuggingFace tokenizer (GPT-2 by default)."""
    name = 'gpt2'

    def __init__(self, model='gpt2'):
        from transformers import AutoTokenizer
        self.tk = AutoTokenizer.from_pretrained(model)
        if self.tk.pad_token is None:
            self.tk.pad_token = self.tk.eos_token
        self.pad_id = self.tk.pad_token_id
        self.bos_id = self.tk.bos_token_id if self.tk.bos_token_id is not None else self.pad_id
        self.eos_id = self.tk.eos_token_id
        self.vocab_size = len(self.tk)

    def encode(self, text, max_len, add_bos_eos=False):
        ids = self.tk(text, truncation=True, max_length=max_len - (2 if add_bos_eos else 0))['input_ids']
        if add_bos_eos:
            ids = [self.bos_id] + ids + [self.eos_id]
        else:
            ids = ids[-max_len:]
        mask = [1] * len(ids) + [0] * (max_len - len(ids))
        ids = ids + [self.pad_id] * (max_len - len(ids))
        return ids, mask

    def state(self):
        return {'model': 'gpt2'}


def make_tokenizer(kind, texts=None, state=None):
    if state is not None:
        return HFTokenizer(state['model']) if 'model' in state else WordTokenizer(state=state)
    if kind in ('gpt2', 'auto'):
        try:
            return HFTokenizer('gpt2')
        except Exception as e:
            if kind == 'gpt2':
                raise
            print(f'[tokenizer] transformers unavailable ({e}); falling back to word-level')
    return WordTokenizer(texts)


# ============================================================================
# DATA  -- context/response pairs, exactly as in the reference Colab pipeline
# ============================================================================
def build_dataset(mwoz_dir, tokenizer_kind='auto', max_dialogues=None):
    def load(split):
        out = []
        for f in sorted(glob.glob(os.path.join(mwoz_dir, split, 'dialogues_*.json'))):
            out += json.load(open(f))
        return out

    pairs = {}
    for split, name in (('train', 'train'), ('dev', 'dev')):
        dlgs = load(split)[:max_dialogues]
        ex = []
        for d in dlgs:
            ctx = []
            for turn in d['turns']:
                if turn['speaker'] == 'SYSTEM':
                    ex.append(('\n'.join(ctx), turn['utterance']))
                ctx.append(f"{turn['speaker']} : {turn['utterance']}")
        pairs[name] = [e for e in ex if e[0]]
        print(f'{name}: {len(dlgs)} dialogues -> {len(pairs[name])} context/response pairs')

    tok = make_tokenizer(tokenizer_kind,
                         texts=[c for c, r in pairs['train']] + [r for c, r in pairs['train']])
    print(f'tokenizer={tok.name} vocab={tok.vocab_size}')

    out = {}
    for name, ex in pairs.items():
        ci, cm, ri = [], [], []
        for c, r in ex:
            a, m = tok.encode(c, CTX_LEN)
            b, _ = tok.encode(r, RSP_LEN, add_bos_eos=True)
            ci.append(a); cm.append(m); ri.append(b)
        out[name] = (torch.tensor(ci), torch.tensor(cm), torch.tensor(ri))
        print(f'  {name}: ctx={tuple(out[name][0].shape)} rsp={tuple(out[name][2].shape)}')

    torch.save({'train': out['train'], 'dev': out['dev'],
                'tok_state': tok.state(), 'vocab_size': tok.vocab_size,
                'pad_id': tok.pad_id}, DATA)


def load_dataset():
    if not os.path.exists(DATA):
        raise SystemExit(f'{DATA} not found -- run: python {__file__} build --mwoz-dir ...')
    return torch.load(DATA)


# ============================================================================
# [CONSTRUCTION: A]   A = S + K in GL(n,R). Fixed, never trained.
# ============================================================================
def fixed_asymmetric(seed):
    g = torch.Generator().manual_seed(seed)
    Sr = torch.randn(NV, NV, generator=g) * 0.4
    Kr = torch.randn(NV, NV, generator=g) * 0.5
    A = 0.5 * (Sr + Sr.T) + 0.5 * (Kr - Kr.T)
    return A / torch.linalg.matrix_norm(A, 2) * 0.8


# ============================================================================
# [THE CONSTRUCTION]  Geodesic encoder.
#
#   t(x) = 0.9 tanh(MLP(x))                belief field on the manifold
#   M(x) = M_base . Cayley(t(x) A)         DEFORMS THE LOCAL INDICATRIX
#   a(x) = M^-T M^-1                       metric tensor field  => THE MANIFOLD
#   F(x,y) = sqrt(a y y) + c.y             deformed Randers norm (c != 0 => irreversible)
#   gamma  from d_x a                      Christoffels: DERIVED
#   G(x,v) Randers spray (gamma, r, s)     DERIVED connection; s = antisymmetric part
#   x' = v ,  v' = -2 G(x,v)               geodesic; velocity rides the curve
#
# Learned here: token embedding (tied, shared with decoder), velocity generator,
# belief MLP, initial points x0.  Everything geometric is derived or fixed.
# ============================================================================
class GeodesicEncoder(nn.Module):
    def __init__(self, seed, emb, ablate_c=False):
        super().__init__()
        torch.manual_seed(seed)
        self.emb = emb                                   # TIED with decoder
        self.gen = nn.Sequential(nn.Linear(D, 64), nn.SiLU(), nn.Linear(64, MVEL * NV))
        self.register_buffer('A', fixed_asymmetric(seed))
        self.register_buffer('Mb', torch.eye(NV) + 0.02 * torch.randn(NV, NV))
        self.belief = nn.Sequential(nn.Linear(NV, 32), nn.SiLU(), nn.Linear(32, 1))
        c = torch.randn(NV, generator=torch.Generator().manual_seed(seed + 9))
        c = torch.zeros(NV) if ablate_c else c / c.norm() * TILT
        self.register_buffer('c', c)                     # Randers tilt = irreversibility
        self.x0 = nn.Parameter(torch.randn(MVEL, NV) * 0.1)

    # ---- [M and THE MANIFOLD] ------------------------------------------------
    def _metric(self, x):
        """a(x) = M(x)^-T M(x)^-1 with M(x) = M_base . Cayley(t(x) A)."""
        t = torch.tanh(self.belief(x)).squeeze(-1) * 0.9
        I = torch.eye(NV, device=x.device, dtype=x.dtype)
        Xm = t.unsqueeze(-1).unsqueeze(-1) * self.A
        M = self.Mb @ torch.linalg.solve(I - 0.5 * Xm, I + 0.5 * Xm)
        Mi = torch.linalg.solve(M, I.expand(x.shape[0], NV, NV))
        return torch.einsum('bki,bkj->bij', Mi, Mi), t

    # ---- [CONNECTION + GEODESIC] --------------------------------------------
    def _step(self, x, v):
        """One derived-geodesic step. Vectorised over (batch * velocities)."""
        B = x.shape[0]
        a, t = self._metric(x)
        ai = torch.linalg.inv(a + 1e-5 * torch.eye(NV, device=x.device, dtype=x.dtype))
        eps = 1e-3
        I = torch.eye(NV, device=x.device, dtype=x.dtype)

        # dt/dx by finite difference through the belief MLP
        dtdx = torch.stack([
            ((torch.tanh(self.belief(
                torch.cat([x[:, :m], x[:, m:m + 1] + eps, x[:, m + 1:]], -1))
             ).squeeze(-1) * 0.9) - t) / eps
            for m in range(NV)], -1)

        # da/dt, then d_m a_ij by the chain rule through t(x)
        def a_of(tv):
            Xm = tv.unsqueeze(-1).unsqueeze(-1) * self.A
            M = self.Mb @ torch.linalg.solve(I - 0.5 * Xm, I + 0.5 * Xm)
            Mi = torch.linalg.solve(M, I.expand(B, NV, NV))
            return torch.einsum('bki,bkj->bij', Mi, Mi)

        da = torch.einsum('bij,bm->bijm', (a_of(t + eps) - a_of(t - eps)) / (2 * eps), dtdx)

        # [CHRISTOFFELS]  gamma^i_jk = 1/2 a^il (d_j a_lk + d_k a_lj - d_l a_jk)
        comb = da.permute(0, 1, 3, 2) + da - da.permute(0, 3, 1, 2)
        gam = 0.5 * torch.einsum('bil,bljk->bijk', ai, comb)

        # [RANDERS]  alpha = sqrt(a v v), beta = c.v, F = alpha + beta
        b = self.c.unsqueeze(0).expand(B, NV)
        alpha = torch.sqrt(torch.clamp(torch.einsum('bi,bij,bj->b', v, a, v), min=1e-8))
        beta = (v * b).sum(-1)
        Fv = (alpha + beta).clamp(min=1e-6)
        l = v / Fv.unsqueeze(-1)

        # [r symmetric, s ANTISYMMETRIC]  s carries F(y) != F(-y); s == 0 iff c == 0
        bcov = -torch.einsum('bkij,bk->bij', gam, b)
        r = 0.5 * (bcov + bcov.transpose(1, 2))
        s = 0.5 * (bcov - bcov.transpose(1, 2))

        # [SPRAY G]  closed-form Randers spray
        G = (0.5 * torch.einsum('bijk,bj,bk->bi', gam, v, v)
             + 0.5 * l * torch.einsum('bjk,bj,bk->b', r, v, v).unsqueeze(-1)
             + alpha.unsqueeze(-1) * torch.einsum(
                 'bij,bj->bi', ai, torch.einsum('bjk,bk->bj', s, v)))

        # [GEODESIC]  x' = v ,  v' = -2G
        xn = x + DTAU * v
        vn = v - 2 * DTAU * G
        n = vn.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        return xn, vn * torch.clamp(n, max=VCLAMP) / n

    # ---- [v_theta, ACCUMULATION, MEMORY] ------------------------------------
    def forward(self, input_ids, attention_mask):
        E = self.emb(input_ids)                                  # [B, T, D]
        B, T, _ = E.shape
        g = self.gen(E).reshape(B, T, MVEL, NV)
        g = g * attention_mask[:, :, None, None].to(g.dtype)      # MASK PADDING

        x = self.x0.unsqueeze(0).expand(B, MVEL, NV).reshape(B * MVEL, NV).contiguous()
        v = torch.zeros(B * MVEL, NV, device=E.device, dtype=E.dtype)

        mem = []
        for t0 in range(0, T, STRIDE):
            gt = g[:, t0:t0 + STRIDE].sum(1).reshape(B * MVEL, NV)
            v = v + gt
            x, v = self._step(x, v)
            # emit the state at this integration step as one memory slot
            mem.append(torch.cat([x, v], -1).reshape(B, MVEL * 2 * NV))
        return torch.stack(mem, 1)                                # [B, NSLOT, DMEM]


# ============================================================================
# BASELINE ENCODER  -- attention, matched width, pooled to the same slot count
# ============================================================================
def rope(x):
    B, T, d = x.shape
    half = d // 2
    fr = torch.exp(-torch.arange(0, half, device=x.device).float() / half * 4.0)
    ang = torch.arange(T, device=x.device).float().unsqueeze(1) * fr.unsqueeze(0)
    cs, sn = torch.cos(ang), torch.sin(ang)
    x1, x2 = x[..., :half], x[..., half:2 * half]
    return torch.cat([x1 * cs - x2 * sn, x1 * sn + x2 * cs, x[..., 2 * half:]], -1)


class AttentionEncoder(nn.Module):
    def __init__(self, seed, emb):
        super().__init__()
        torch.manual_seed(seed)
        self.emb = emb                                   # TIED with decoder
        self.ln = nn.LayerNorm(D)
        self.attn = nn.MultiheadAttention(D, 4, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(D, 2 * D), nn.GELU(), nn.Linear(2 * D, D))
        self.proj = nn.Linear(D, DMEM)

    def forward(self, input_ids, attention_mask):
        h = self.emb(input_ids)
        hn = rope(self.ln(h))
        kpm = (attention_mask == 0)
        o, _ = self.attn(hn, hn, hn, key_padding_mask=kpm, need_weights=False)
        h = h + o
        h = h + self.mlp(h)
        h = h * attention_mask.unsqueeze(-1).to(h.dtype)
        B, T, _ = h.shape
        # strided mean-pool to NSLOT slots so the decoder sees the same budget
        h = h.reshape(B, T // STRIDE, STRIDE, D).mean(2)
        return self.proj(h)                                       # [B, NSLOT, DMEM]


# ============================================================================
# SHARED DECODER  -- identical for both encoders
# ============================================================================
class Decoder(nn.Module):
    def __init__(self, seed, emb, vocab_size):
        super().__init__()
        torch.manual_seed(seed + 777)
        self.emb = emb                                   # TIED with encoder
        self.mem_in = nn.Linear(DMEM, D)
        layer = nn.TransformerDecoderLayer(D, DEC_HEADS, 4 * D, batch_first=True,
                                           dropout=0.0, norm_first=True)
        self.dec = nn.TransformerDecoder(layer, DEC_LAYERS)
        self.ln = nn.LayerNorm(D)
        self.out = nn.Linear(D, vocab_size, bias=False)
        self.out.weight = emb.weight                     # weight tying

    def forward(self, memory, tgt_in):
        m = self.mem_in(memory)
        h = rope(self.emb(tgt_in))
        T = tgt_in.shape[1]
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=tgt_in.device), 1)
        h = self.dec(h, m, tgt_mask=causal)
        return self.out(self.ln(h))


class Seq2Seq(nn.Module):
    """Encoder + shared decoder + tied embedding table."""
    def __init__(self, encoder_kind, seed, vocab_size, pad_id, ablate_c=False):
        super().__init__()
        torch.manual_seed(seed)
        emb = nn.Embedding(vocab_size, D, padding_idx=pad_id)
        nn.init.normal_(emb.weight, std=0.02)
        with torch.no_grad():
            emb.weight[pad_id].zero_()
        self.encoder = (GeodesicEncoder(seed, emb, ablate_c)
                        if encoder_kind == 'geodesic' else AttentionEncoder(seed, emb))
        self.decoder = Decoder(seed, emb, vocab_size)
        self.pad_id = pad_id

    def forward(self, ctx_ids, ctx_mask, rsp_ids):
        mem = self.encoder(ctx_ids, ctx_mask)
        logits = self.decoder(mem, rsp_ids[:, :-1])
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                               rsp_ids[:, 1:].reshape(-1),
                               ignore_index=self.pad_id)
        return logits, loss


# ============================================================================
# TRAIN / EVAL
# ============================================================================
def iterate(tensors, bs, rng, shuffle=True):
    n = len(tensors[0])
    idx = torch.randperm(n, generator=rng) if shuffle else torch.arange(n)
    for i in range(0, n - bs + 1, bs):
        j = idx[i:i + bs]
        yield [t[j] for t in tensors]


def run(encoder, seed, steps, bs=16, lr=3e-4, ablate_c=False, eval_batches=40, device='cpu'):
    d = load_dataset()
    tr, dv = d['train'], d['dev']
    torch.manual_seed(1234)                              # identical init stream
    m = Seq2Seq(encoder, seed, d['vocab_size'], d['pad_id'], ablate_c).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=lr)
    rng = torch.Generator().manual_seed(seed)

    tag = f'{encoder}{"_c0" if ablate_c else ""}'
    ck = f'seq2seq_{tag}_{seed}.pt'
    it = 0
    if os.path.exists(ck):
        st = torch.load(ck)
        m.load_state_dict(st['m']); opt.load_state_dict(st['o']); it = st['it']
        print(f'resumed at {it}')

    m.train()
    t0, target = time.time(), it + steps
    while it < target:
        for ci, cm, ri in iterate(tr, bs, rng):
            _, loss = m(ci.to(device), cm.to(device), ri.to(device))
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step(); it += 1
            if it % 100 == 0:
                print(f'  it={it} loss={loss.item():.4f} ({time.time()-t0:.0f}s)', flush=True)
            if it >= target:
                break
    torch.save({'m': m.state_dict(), 'o': opt.state_dict(), 'it': it}, ck)

    m.eval()
    tot, n = 0.0, 0
    with torch.no_grad():
        for k, (ci, cm, ri) in enumerate(iterate(dv, bs, rng, shuffle=False)):
            _, loss = m(ci.to(device), cm.to(device), ri.to(device))
            tot += loss.item(); n += 1
            if k + 1 >= eval_batches:
                break
    ce = tot / max(n, 1)

    res = json.load(open(RESULTS)) if os.path.exists(RESULTS) else {}
    res.setdefault(tag, {})[str(seed)] = {'ce': ce, 'ppl': math.exp(ce), 'it': it}
    json.dump(res, open(RESULTS, 'w'), indent=1)

    enc_p = sum(p.numel() for n_, p in m.encoder.named_parameters() if not n_.startswith('emb'))
    print(f'[{tag} seed={seed}] it={it} val_CE={ce:.4f} ppl={math.exp(ce):.2f} '
          f'encoder_params(excl. emb)={enc_p:,} ({time.time()-t0:.0f}s)')
    return ce


def summary():
    if not os.path.exists(RESULTS):
        raise SystemExit('no results yet')
    res = json.load(open(RESULTS))
    print(f'{"encoder":<18}{"seeds (val CE)":<28}{"mean +/- std":<20}{"ppl"}')
    print('-' * 74)
    for tag, seeds in res.items():
        v = [s['ce'] for _, s in sorted(seeds.items())]
        mu = sum(v) / len(v)
        sd = (sum((x - mu) ** 2 for x in v) / len(v)) ** 0.5
        print(f'{tag:<18}{"  ".join(f"{x:.3f}" for x in v):<28}'
              f'{mu:.4f} +/- {sd:.4f}     {math.exp(mu):.2f}')


# ============================================================================
# if __name__ == '__main__':
#     p = argparse.ArgumentParser()
#     sub = p.add_subparsers(dest='cmd', required=True)

#     b = sub.add_parser('build')
#     b.add_argument('--mwoz-dir', required=True)
#     b.add_argument('--tokenizer', default='auto', choices=['auto', 'gpt2', 'word'])
#     b.add_argument('--max-dialogues', type=int, default=None)

#     r = sub.add_parser('run')
#     r.add_argument('--encoder', choices=['geodesic', 'attention'], required=True)
#     r.add_argument('--seed', type=int, default=0)
#     r.add_argument('--steps', type=int, default=2000)
#     r.add_argument('--batch-size', type=int, default=16)
#     r.add_argument('--lr', type=float, default=3e-4)
#     r.add_argument('--ablate-c', action='store_true')
#     r.add_argument('--device', default='cuda' if torch.cuda.is_available() else 'cpu')

#     sub.add_parser('summary')

#     a = p.parse_args()
#     if a.cmd == 'build':
#         build_dataset(a.mwoz_dir, a.tokenizer, a.max_dialogues)
#     elif a.cmd == 'run':
#         run(a.encoder, a.seed, a.steps, a.batch_size, a.lr, a.ablate_c, device=a.device)
#     else:
#         summary()



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Seq2Seq(
    encoder_kind="geodesic",
    seed=0,
    vocab_size=tokenizer.vocab_size,
    pad_id=tokenizer.pad_token_id
).to(device)

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

In [ ]:
EPOCHS = 10

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for batch in train_loader:

        ctx_ids = batch["ctx_ids"].to(device)
        ctx_mask = batch["ctx_mask"].to(device)
        rsp_ids = batch["rsp_ids"].to(device)

        logits, loss = model(
            ctx_ids,
            ctx_mask,
            rsp_ids
        )

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1} | "
        f"Loss = {total_loss/len(train_loader):.4f}"
    )

In [ ]:
!nvidia-smi

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)
print(torch.cuda.get_device_name(0))

In [ ]:
batch = next(iter(train_loader))

ctx = batch["ctx_ids"].to(device)
mask = batch["ctx_mask"].to(device)
rsp = batch["rsp_ids"].to(device)

logits, loss = model(ctx, mask, rsp)

print(logits.shape)
print(loss.item())

In [ ]:
EPOCHS = 1

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):

        ctx_ids = batch["ctx_ids"].to(device)
        ctx_mask = batch["ctx_mask"].to(device)
        rsp_ids = batch["rsp_ids"].to(device)

        logits, loss = model(ctx_ids, ctx_mask, rsp_ids)

        optimizer.zero_grad()
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 20 == 0:
            print(
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Loss = {loss.item():.4f}"
            )

    print(f"Epoch Loss = {total_loss / len(train_loader):.4f}")

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "tokenizer_name": "gpt2",
}, "geodesic_seq2seq.pt")

In [ ]:
from google.colab import files
files.download("geodesic_seq2seq.pt")

In [ ]:
# Loading 1 epoch trained model
import torch
checkpoint = torch.load("geodesic_seq2seq.pt", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Model loaded successfully!")

In [ ]:
# Evaluating the loss

model.eval()

total_loss = 0

with torch.no_grad():

    for batch in train_loader:

        ctx_ids = batch["ctx_ids"].to(device)
        ctx_mask = batch["ctx_mask"].to(device)
        rsp_ids = batch["rsp_ids"].to(device)

        _, loss = model(
            ctx_ids,
            ctx_mask,
            rsp_ids
        )

        total_loss += loss.item()

avg_loss = total_loss / len(train_loader)

print("Average Loss:", avg_loss)

In [ ]:
# Generation

import torch

@torch.no_grad()
def generate_response(model, tokenizer, prompt, device, max_new_tokens=50):

    model.eval()

    # Encode context
    enc = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256
    )

    ctx_ids = enc["input_ids"].to(device)
    ctx_mask = enc["attention_mask"].to(device)

    # Encode context once
    memory = model.encoder(ctx_ids, ctx_mask)

    # Start with GPT-2 EOS token (used as BOS)
    generated = torch.tensor([[tokenizer.eos_token_id]], device=device)

    for _ in range(max_new_tokens):

        logits = model.decoder(memory, generated)

        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)

        generated = torch.cat([generated, next_token], dim=1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(
        generated.squeeze(),
        skip_special_tokens=True
    )

In [ ]:
prompt = """USER: I want somewhere pricey in downtown."""

response = generate_response(
    model,
    tokenizer,
    prompt,
    device
)

print(response)

In [ ]:
tests = [
    "USER: I want a luxury spaceship hotel on Mars."
]

for prompt in tests:

    print("="*70)
    print(prompt)
    print()
    print(generate_response(model, tokenizer, prompt, device))
    print()

In [ ]:
tests = [
    "USER: I'd like somewhere pricey to stay downtown."
]

for prompt in tests:

    print("="*70)
    print(prompt)
    print()
    print(generate_response(model, tokenizer, prompt, device))
    print()



In [ ]:
import re
import matplotlib.pyplot as plt

# Read the training log
with open("Pasted text(5).txt", "r") as f:
    text = f.read()

# Extract batch number and loss
matches = re.findall(r"Batch (\d+)/\d+ \| Loss = ([0-9.]+)", text)

batches = [int(x[0]) for x in matches]
losses = [float(x[1]) for x in matches]

plt.figure(figsize=(12,6))
plt.plot(batches, losses, linewidth=2)

plt.title("Training Loss of Geometry-Based Seq2Seq Model", fontsize=18)
plt.xlabel("Training Batch", fontsize=14)
plt.ylabel("Cross Entropy Loss", fontsize=14)

plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig("training_loss.png", dpi=300)

plt.show()